<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/main/RA3_LAB4/EXPERIENCIA_4%20/RA3_Lab_N%C2%B04_EXP2_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/alex/RA3_LAB4/EXPERIENCIA_4/RA3_Lab_N%C2%B04_EXP4_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiencia 4 - Identificación y Reducción de Orden de Sistemas Dinámicos (DEAP)

**IC415 - Inteligencia Computacional - RA3 / Laboratorio N 4**

Se identifica, mediante un algoritmo genético sobre DEAP, una función de transferencia de orden
reducido que reproduce la respuesta al escalón medida de un sistema de octavo orden. El individuo
es un vector de coeficientes reales; el fitness mide la discrepancia (RMSE) entre la respuesta
simulada y la medida. Se exploran los órdenes n ∈ {2, 3, 4, 5}, combinando la búsqueda global del
AG con un refinamiento local final, y se analiza el compromiso entre fidelidad de ajuste y
complejidad del modelo.

El contenido del notebook es deliberadamente conciso: el código y las figuras son la evidencia
técnica reproducible; la interpretación extensa corresponde al informe.

## 0. Configuración del entorno

Instalación de dependencias, importaciones y fijado global de semillas. La simulación de la
respuesta al escalón —el componente más costoso— se resuelve discretizando la transferencia (ZOH)
y aplicando un filtro lineal, mucho más rápido que una integración genérica; la librería `control`
se reserva para presentar los modelos finales (función de transferencia, Bode, polos y ceros).

In [ ]:
# Instalación silenciosa (idempotente)
import importlib, subprocess, sys
for paq in ["deap", "control"]:
    if importlib.util.find_spec(paq) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paq], check=False)

import os, time, random, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal, optimize
from deap import base, creator, tools
import control as ct

# Reproducibilidad global
SEMILLA_BASE = 42
random.seed(SEMILLA_BASE); np.random.seed(SEMILLA_BASE)

FIGS_DIR = "figuras_informe_exp4"
os.makedirs(FIGS_DIR, exist_ok=True)
print("NumPy:", np.__version__, "| figuras en:", os.path.abspath(FIGS_DIR))

NumPy: 2.4.6 | figuras en: /tmp/figuras_informe_exp4


## 1. Datos: respuesta medida del sistema

La respuesta al escalón del sistema real (de octavo orden) está tabulada en `sistema.csv`
(columnas tiempo y amplitud). Se la caracteriza antes de optimizar: ganancia de estado
estacionario, sobreimpulso y tiempo de pico fijan las metas que el modelo reducido debe
reproducir.

In [ ]:
# Carga: usa el archivo local si está; si no, lo descarga (entorno Colab)
RUTA_CSV = "sistema.csv"
if not os.path.exists(RUTA_CSV):
    url = ("https://drive.google.com/uc?export=download&id=1DyL84D0OuOAdyATOT9TfzCb7iAIcYhOh&confirm=t")
    subprocess.run(["wget", "-q", "-c", "--no-check-certificate", url, "-O", RUTA_CSV], check=False)

datos = pd.read_csv(RUTA_CSV)
T = datos["t"].to_numpy(); Y = datos["Amplitud"].to_numpy()
DT = T[1] - T[0]; STEP = np.ones_like(T)

ss = Y[-1]; pico = Y.max(); t_pico = T[Y.argmax()]
overshoot = 100 * (pico - ss) / ss
print(f"Puntos: {len(T)} | t: {T[0]:.2f}..{T[-1]:.2f} s | dt = {DT:.3f} s")
print(f"Valor de estado estacionario: {ss:.4f}")
print(f"Pico: {pico:.4f} en t = {t_pico:.2f} s  ->  sobreimpulso = {overshoot:.1f}%")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(T, Y, lw=1.6)
ax.set_xlabel("Tiempo (s)"); ax.set_ylabel("Amplitud")
ax.set_title("Respuesta al escalón del sistema real (8.º orden)"); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(FIGS_DIR, "fig_respuesta_medida.png"), dpi=150)
plt.close(fig); print("Guardado: fig_respuesta_medida.png")

Puntos: 1000 | t: 0.00..19.98 s | dt = 0.020 s
Valor de estado estacionario: 1.0007
Pico: 2.2032 en t = 0.46 s  ->  sobreimpulso = 120.2%


Guardado: fig_respuesta_medida.png


## 2. Representación, modelo y simulación

Cada individuo es un **vector de coeficientes reales**. Para un orden $n$ se codifican los $n$
coeficientes del numerador y los $n$ coeficientes libres del denominador (mónico, con $s^n$ de
coeficiente unitario fijado por construcción), de modo que la función de transferencia es propia:

$$H(s)=\dfrac{w_{n-1}s^{n-1}+\dots+w_1 s+w_0}{s^{n}+x_{n-1}s^{n-1}+\dots+x_1 s+x_0}$$

La respuesta al escalón se obtiene discretizando $H(s)$ por retención de orden cero al paso de
muestreo de los datos y filtrando un escalón unitario. Se valida el simulador rápido contra un
modelo de segundo orden de referencia: debe reproducir el RMSE esperado.

In [ ]:
def construir(genes, n):
    '''genes (longitud 2n) -> (num, den). den es mónico: [1, ...].'''
    num = list(genes[:n]); den = [1.0] + list(genes[n:])
    return num, den

def simular(num, den):
    '''Respuesta al escalón por discretización ZOH + filtrado lineal (rápida).'''
    dnum, dden, _ = signal.cont2discrete((num, den), DT, method="zoh")
    return signal.lfilter(dnum.ravel(), dden.ravel(), STEP)

def es_estable(den):
    '''Estable si todas las raíces del denominador tienen parte real negativa.'''
    return np.all(np.real(np.roots(den)) < 0)

def rmse(y_sim):
    return float(np.sqrt(np.mean((Y - y_sim) ** 2)))

# Validación: modelo de 2.º orden de referencia
num_ref, den_ref = [16.42, 5.352], [1.0, 6.458, 5.352]
y_ref = simular(num_ref, den_ref)
print("Modelo de referencia (2.º orden)  RMSE =", round(rmse(y_ref), 5), "(esperado ~0.0132)")
print("¿Estable?", es_estable(den_ref), "| polos:", np.round(np.roots(den_ref), 3))

Modelo de referencia (2.º orden)  RMSE = 0.01317 (esperado ~0.0132)
¿Estable? True | polos: [-5.482 -0.976]


## 3. Función de fitness y manejo de la estabilidad

El fitness es el RMSE entre la respuesta simulada y la medida (a minimizar). Una función de
transferencia es válida sólo si es estable; un individuo con algún polo de parte real no negativa
produciría una respuesta que diverge y carece de sentido físico. Esos individuos se penalizan con
un valor de fitness grande en lugar de descartarlos o repararlos: la penalización es simple, no
requiere proyectar coeficientes a la región estable y deja que la presión selectiva expulse de
forma natural a las soluciones inestables, conservando además información de gradiente hacia la
zona factible. Un contador registra qué fracción de las evaluaciones cae en esa penalización.

In [ ]:
PENAL = 1e3
stats_eval = {"total": 0, "penal": 0, "t_acumulado": 0.0}

def make_eval(n):
    def evaluate(ind):
        t0 = time.time()
        num, den = construir(ind, n)
        val = PENAL
        if es_estable(den):
            try:
                y_sim = simular(num, den)
                if np.all(np.isfinite(y_sim)):
                    val = rmse(y_sim)
            except Exception:
                val = PENAL
        stats_eval["total"] += 1
        if val >= PENAL: stats_eval["penal"] += 1
        stats_eval["t_acumulado"] += time.time() - t0
        return (val,)
    return evaluate

## 4. Operadores genéticos para representación real

Al ser los individuos vectores de números reales (no cadenas binarias ni permutaciones), los
operadores deben moverse de forma continua dentro de límites: se emplean cruce binario simulado
acotado (SBX) y mutación polinómica acotada, ambos parametrizados por un índice de distribución
que controla cuán cerca de los padres se generan los hijos, junto con selección por torneo. A
diferencia de la representación binaria (cruce de punto, *bit-flip*) o de la permutación (cruce de
orden, intercambio), estos operadores explotan la noción de distancia y respetan por construcción
el dominio admisible de cada coeficiente.

In [ ]:
if not hasattr(creator, "FitMin"):
    creator.create("FitMin", base.Fitness, weights=(-1.0,))
    creator.create("Indiv", list, fitness=creator.FitMin)

LOW, UP = 0.0, 30.0   # dominio de los coeficientes (positivos: condición necesaria de estabilidad)

def construir_toolbox(n):
    tb = base.Toolbox()
    tb.register("attr", random.uniform, LOW, UP)
    tb.register("individual", tools.initRepeat, creator.Indiv, tb.attr, 2 * n)
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("evaluate", make_eval(n))
    tb.register("mate", tools.cxSimulatedBinaryBounded, eta=15.0, low=LOW, up=UP)
    tb.register("mutate", tools.mutPolynomialBounded, eta=20.0, low=LOW, up=UP, indpb=0.2)
    tb.register("select", tools.selTournament, tournsize=3)
    return tb

## 5. Motor evolutivo y refinamiento local

El AG explora globalmente el espacio de coeficientes con elitismo y parada por estancamiento (no
hay óptimo conocido de antemano, a diferencia de un problema de factibilidad). Como la función a
ajustar es continua, sobre el mejor individuo de cada corrida se aplica un **refinamiento local**
(Nelder-Mead) que pule los coeficientes; esta hibridación es la que permite alcanzar ajustes de
alta fidelidad de forma reproducible, especialmente en los órdenes mayores.

In [ ]:
def run_ga(n, seed, pop_size=300, max_gen=60, cxpb=0.7, mutpb=0.3, n_elite=2, stagn=15):
    random.seed(seed); np.random.seed(seed)
    tb = construir_toolbox(n)
    pop = tb.population(pop_size)
    for ind in pop: ind.fitness.values = tb.evaluate(ind)
    hof = tools.HallOfFame(1); hof.update(pop)
    best = hof[0].fitness.values[0]; ult = 0; mins = [best]
    t0 = time.time()
    for gen in range(1, max_gen + 1):
        elite = [tb.clone(x) for x in tools.selBest(pop, n_elite)]
        off = [tb.clone(x) for x in tb.select(pop, pop_size - n_elite)]
        for a, b in zip(off[::2], off[1::2]):
            if random.random() < cxpb:
                tb.mate(a, b); del a.fitness.values; del b.fitness.values
        for m in off:
            if random.random() < mutpb:
                tb.mutate(m); del m.fitness.values
        for i in [x for x in off if not x.fitness.valid]:
            i.fitness.values = tb.evaluate(i)
        pop = elite + off; hof.update(pop)
        cur = hof[0].fitness.values[0]; mins.append(cur)
        if cur < best - 1e-9: best = cur; ult = gen
        if gen - ult >= stagn: break
    return dict(n=n, seed=seed, rmse_ga=best, gen=gen,
                tiempo_ga=time.time() - t0, genes=list(hof[0]), mins=mins)

def pulir(genes, n, maxiter=3000):
    '''Refinamiento local de los coeficientes (Nelder-Mead) minimizando el RMSE.'''
    def obj(x):
        num, den = construir(x, n)
        if not es_estable(den): return PENAL
        try:
            y_sim = simular(num, den)
            return rmse(y_sim) if np.all(np.isfinite(y_sim)) else PENAL
        except Exception:
            return PENAL
    res = optimize.minimize(obj, np.array(genes, float), method="Nelder-Mead",
                            options=dict(maxiter=maxiter, xatol=1e-7, fatol=1e-10))
    return list(res.x), float(res.fun)

print("Motor evolutivo y refinamiento definidos.")

Motor evolutivo y refinamiento definidos.


## 6. Experimentación: barrido de órdenes

Para cada orden $n \in \{2,3,4,5\}$ se ejecutan varias semillas independientes; de cada corrida se
toma el mejor individuo del AG y se lo refina localmente. Se conserva, por orden, el mejor modelo
hallado y se consolidan los resultados.

In [ ]:
ORDENES = [2, 3, 4, 5]
R = 5
stats_eval.update(total=0, penal=0, t_acumulado=0.0)

corridas = []; mejores = {}
t_total0 = time.time()
for n in ORDENES:
    print(f"=== Orden n={n} ===")
    for r in range(R):
        seed = SEMILLA_BASE + r
        res = run_ga(n, seed)
        genes_p, rmse_p = pulir(res["genes"], n)
        res.update(genes_pulido=genes_p, rmse=rmse_p)
        corridas.append(res)
        if n not in mejores or rmse_p < mejores[n]["rmse"]:
            mejores[n] = res
        print(f"  seed={seed}: RMSE_AG={res['rmse_ga']:.5f} -> pulido={rmse_p:.6f} "
              f"(gen={res['gen']}, t={res['tiempo_ga']:.1f}s)")
t_total = time.time() - t_total0

df = pd.DataFrame([{k: v for k, v in c.items() if k not in ("mins", "genes", "genes_pulido")}
                   for c in corridas])
df.to_csv(os.path.join(FIGS_DIR, "resultados_corridas_exp4.csv"), index=False)
print(f"\nTiempo total del barrido: {t_total:.1f} s")

=== Orden n=2 ===


  seed=42: RMSE_AG=0.04585 -> pulido=0.005743 (gen=60, t=6.9s)


  seed=43: RMSE_AG=0.04438 -> pulido=0.005743 (gen=60, t=6.8s)


  seed=44: RMSE_AG=0.01146 -> pulido=0.005743 (gen=60, t=6.8s)


  seed=45: RMSE_AG=0.01469 -> pulido=0.005743 (gen=60, t=7.0s)


...

  seed=42: RMSE_AG=0.00771 -> pulido=0.000295 (gen=60, t=9.2s)


  seed=43: RMSE_AG=0.03919 -> pulido=0.000639 (gen=60, t=9.1s)


  seed=44: RMSE_AG=0.03756 -> pulido=0.002914 (gen=60, t=8.6s)


  seed=45: RMSE_AG=0.02445 -> pulido=0.000845 (gen=60, t=8.8s)


  seed=46: RMSE_AG=0.02046 -> pulido=0.000300 (gen=60, t=9.0s)

Tiempo total del barrido: 200.3 s


## 7. Análisis

### 7.1 Fidelidad frente a complejidad del modelo

In [ ]:
filas = []
for n in ORDENES:
    sub = df[df["n"] == n]
    filas.append({
        "Orden n": n,
        "Parámetros libres (2n)": 2 * n,
        "RMSE mejor": f"{mejores[n]['rmse']:.2e}",
        "RMSE medio (pulido)": f"{sub['rmse'].mean():.2e}",
        "Tiempo medio AG (s)": f"{sub['tiempo_ga'].mean():.2f}",
    })
tabla_ord = pd.DataFrame(filas)
print(tabla_ord.to_string(index=False))
tabla_ord.to_csv(os.path.join(FIGS_DIR, "tabla_ordenes_exp4.csv"), index=False)

fig, ax1 = plt.subplots(figsize=(7, 4.2))
rmses = [mejores[n]["rmse"] for n in ORDENES]
ax1.plot(ORDENES, rmses, "o-", color="C0", lw=1.8)
ax1.set_yscale("log"); ax1.set_xlabel("Orden del modelo reducido (n)")
ax1.set_ylabel("RMSE del mejor modelo (log)", color="C0")
ax1.set_xticks(ORDENES); ax1.grid(alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(ORDENES, [2 * n for n in ORDENES], "s--", color="C3", lw=1.4)
ax2.set_ylabel("Parámetros libres (2n)", color="C3")
ax1.set_title("Compromiso fidelidad–complejidad")
fig.tight_layout(); fig.savefig(os.path.join(FIGS_DIR, "fig_orden_vs_error.png"), dpi=150)
plt.close(fig); print("Guardado: fig_orden_vs_error.png")

 Orden n  Parámetros libres (2n) RMSE mejor RMSE medio (pulido) Tiempo medio AG (s)
       2                       4   5.74e-03            5.74e-03                6.89
       3                       6   3.11e-04            3.11e-04                7.97
       4                       8   2.27e-05            2.20e-04                8.25
       5                      10   2.95e-04            9.98e-04                8.93


Guardado: fig_orden_vs_error.png


### 7.2 Ajuste de los mejores modelos por orden

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(T, Y, "k", lw=2.2, label="Sistema real (medido)")
for n in ORDENES:
    num, den = construir(mejores[n]["genes_pulido"], n)
    ax.plot(T, simular(num, den), lw=1.2, label=f"n={n} (RMSE {mejores[n]['rmse']:.1e})")
ax.set_xlim(0, 6); ax.set_xlabel("Tiempo (s)"); ax.set_ylabel("Amplitud")
ax.set_title("Ajuste de los modelos reducidos a la respuesta medida")
ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIGS_DIR, "fig_ajuste_ordenes.png"), dpi=150)
plt.close(fig); print("Guardado: fig_ajuste_ordenes.png")

Guardado: fig_ajuste_ordenes.png


### 7.3 Métricas de fidelidad dinámica

El RMSE resume el error punto a punto, pero no garantiza por sí solo que el modelo reproduzca
rasgos dinámicos clave. Para el mejor modelo global se contrastan el error de estado estacionario,
el sobreimpulso y el tiempo de pico frente a los del sistema real.

In [ ]:
def metricas_dinamicas(y):
    ss_ = y[-1]; pico_ = y.max(); tp_ = T[y.argmax()]
    return ss_, pico_, tp_, 100 * (pico_ - ss_) / ss_

# mejor modelo global (menor RMSE entre todos los órdenes)
n_best = min(ORDENES, key=lambda n: mejores[n]["rmse"])
num_b, den_b = construir(mejores[n_best]["genes_pulido"], n_best)
y_b = simular(num_b, den_b)
ss_r, pico_r, tp_r, ov_r = metricas_dinamicas(Y)
ss_m, pico_m, tp_m, ov_m = metricas_dinamicas(y_b)

comp = pd.DataFrame([
    {"Métrica": "Estado estacionario", "Sistema real": f"{ss_r:.4f}", f"Modelo n={n_best}": f"{ss_m:.4f}"},
    {"Métrica": "Sobreimpulso (%)",   "Sistema real": f"{ov_r:.1f}",  f"Modelo n={n_best}": f"{ov_m:.1f}"},
    {"Métrica": "Tiempo de pico (s)", "Sistema real": f"{tp_r:.2f}",  f"Modelo n={n_best}": f"{tp_m:.2f}"},
    {"Métrica": "RMSE global",        "Sistema real": "0",            f"Modelo n={n_best}": f"{mejores[n_best]['rmse']:.2e}"},
])
print(f"Mejor modelo global: orden n={n_best}")
print(comp.to_string(index=False))
comp.to_csv(os.path.join(FIGS_DIR, "tabla_metricas_dinamicas_exp4.csv"), index=False)

Mejor modelo global: orden n=4
            Métrica Sistema real Modelo n=4
Estado estacionario       1.0007     1.0008
   Sobreimpulso (%)        120.2      120.2
 Tiempo de pico (s)         0.46       0.46
        RMSE global            0   2.27e-05


### 7.4 Estabilidad y costo computacional

In [ ]:
# Estabilidad: fracción de evaluaciones penalizadas (individuos inestables)
frac_penal = 100 * stats_eval["penal"] / max(1, stats_eval["total"])
# Costo: la simulación domina el tiempo de cómputo
t_por_eval = 1000 * stats_eval["t_acumulado"] / max(1, stats_eval["total"])
print(f"Evaluaciones de fitness totales : {stats_eval['total']:,}")
print(f"  penalizadas por inestabilidad : {stats_eval['penal']:,}  ({frac_penal:.1f}%)")
print(f"Tiempo medio por evaluación     : {t_por_eval:.3f} ms")
print(f"Tiempo total en evaluaciones    : {stats_eval['t_acumulado']:.1f} s "
      f"({100*stats_eval['t_acumulado']/t_total:.0f}% del barrido)")

Evaluaciones de fitness totales : 288,491
  penalizadas por inestabilidad : 6,424  (2.2%)
Tiempo medio por evaluación     : 0.532 ms
Tiempo total en evaluaciones    : 153.4 s (77% del barrido)


### 7.5 Convergencia y modelo final (función de transferencia, Bode y polos–ceros)

In [ ]:
# Curvas de convergencia (mejor semilla por orden)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for n in ORDENES:
    mins = mejores[n]["mins"]
    ax.plot(range(len(mins)), mins, lw=1.4, label=f"n={n}")
ax.set_yscale("log"); ax.set_xlabel("Generación"); ax.set_ylabel("RMSE (mejor, log)")
ax.set_title("Convergencia del AG por orden (antes del pulido)"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIGS_DIR, "fig_convergencia_exp4.png"), dpi=150)
plt.close(fig); print("Guardado: fig_convergencia_exp4.png")

# Modelo final con la librería de control: TF, Bode y polos-ceros
sys_best = ct.tf(num_b, den_b)
print(f"\nFunción de transferencia del mejor modelo (n={n_best}):")
print(sys_best)

fig = plt.figure(figsize=(7, 5)); ct.bode_plot(sys_best)
plt.suptitle(f"Diagrama de Bode - mejor modelo (n={n_best})")
fig.tight_layout(); fig.savefig(os.path.join(FIGS_DIR, "fig_bode_best.png"), dpi=150)
plt.close(fig)

polos = np.roots(den_b); ceros = np.roots(num_b)
fig, ax = plt.subplots(figsize=(5.2, 5))
ax.scatter(np.real(polos), np.imag(polos), marker="x", s=80, color="C3", label="Polos")
if len(ceros): ax.scatter(np.real(ceros), np.imag(ceros), marker="o", s=70,
                          facecolors="none", edgecolors="C0", label="Ceros")
ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
ax.set_xlabel("Re"); ax.set_ylabel("Im"); ax.grid(alpha=0.3); ax.legend()
ax.set_title(f"Polos y ceros - mejor modelo (n={n_best})")
fig.tight_layout(); fig.savefig(os.path.join(FIGS_DIR, "fig_polos_ceros_best.png"), dpi=150)
plt.close(fig); print("Guardado: fig_bode_best.png, fig_polos_ceros_best.png")

Guardado: fig_convergencia_exp4.png

Función de transferencia del mejor modelo (n=4):
<TransferFunction>: sys[0]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']

    18.01 s^3 + 163.3 s^2 + 299.2 s + 79.6
  -------------------------------------------
  s^4 + 16.53 s^3 + 85.14 s^2 + 149 s + 79.54


Guardado: fig_bode_best.png, fig_polos_ceros_best.png


## 8. Viabilidad de implementación

El modelo reducido obtenido es de orden bajo, lo que lo vuelve apto para su implementación en un
microcontrolador: una transferencia de orden $n$ discretizada se evalúa como una ecuación en
diferencias con del orden de $2n$ multiplicaciones-acumulaciones por ciclo de control, muy por
debajo de las del modelo original de octavo orden. El equilibrio entre fidelidad y costo —y la
decisión de hasta qué orden conviene subir— se discute en el informe a partir de las tablas y
figuras anteriores.

## 9. Exportación de figuras para el informe

Se empaquetan figuras y tablas en un único ZIP descargable.

In [ ]:
zip_path = "figuras_informe_Exp4.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for nombre in sorted(os.listdir(FIGS_DIR)):
        zf.write(os.path.join(FIGS_DIR, nombre), arcname=nombre)
print("Contenido del ZIP:")
with zipfile.ZipFile(zip_path) as zf:
    for n in zf.namelist(): print("  -", n)

try:
    from google.colab import files  # type: ignore
    files.download(zip_path)
except Exception:
    print("\nFuera de Colab: ZIP disponible en", os.path.abspath(zip_path))

Contenido del ZIP:
  - fig_ajuste_ordenes.png
  - fig_bode_best.png
  - fig_convergencia_exp4.png
  - fig_orden_vs_error.png
  - fig_polos_ceros_best.png
  - fig_respuesta_medida.png
  - resultados_corridas_exp4.csv
  - tabla_metricas_dinamicas_exp4.csv
  - tabla_ordenes_exp4.csv

Fuera de Colab: ZIP disponible en /tmp/figuras_informe_Exp4.zip
